![databricks_academy_logo.png](../Includes/images/databricks_academy_logo.png "databricks_academy_logo.png")

# Lab - Creating Tools and Testing in the Playground
In this demo, you will explore how to create and register agent tools in Databricks to streamline customer service workflows. By combining structured tools (such as retrieving a customer’s interaction history) with unstructured tools (like searching product documentation through vector search), you will see how AI can automate common support tasks. The exercise walks you through building Unity Catalog functions, testing them in the AI Playground, and observing how different models leverage your tools to provide accurate, context-aware answers. 

### Learning Objectives
_By the end of this lab, you will be able to:_
- Create structured tools using Unity Catalog functions to retrieve customer interaction history.
- Build unstructured tools that leverage vector search for product documentation retrieval.
- Test and validate custom tools in Databricks SQL.
- Register and enable tools for use within the AI Playground.
- Interact with an AI agent that uses your tools to handle real-world customer service workflows.
- Compare model responses and (optionally) export your agent setup for deployment.

## Lab Scenario: Empowering Customer Service Representatives

Customer service representatives at your company handle a high volume of support requests every day. To provide excellent service, they need to quickly review each customer’s past support history and access detailed product documentation to answer questions and resolve issues. However, searching for this information manually is time-consuming and can lead to delays or inconsistent answers. As a data and AI specialist, your goal is to empower the support team by building an intelligent agent that streamlines their workflow. This agent will allow customer service reps to instantly retrieve a customer’s interaction history and search product manuals for troubleshooting steps or technical details. By automating these tasks, you’ll help the team respond faster, improve accuracy, and deliver a better customer experience. In this lab, you’ll design, build, and test this AI-powered assistant using real customer service datasets and Databricks tools.

## Important: Select Environment 4
The cells below may not work in other environments. To choose environment 4: 
1. Click the ![environment.png](../../Includes/images/environment.png "environment.png") button on the right sidebar
1. Open the **Environment version** dropdown
1. Select **4**

## REQUIRED - Setting required variables

Run the following cell to configure some python variables we will be using in the rest of the notebook

In [0]:
####################################################################################
# Set python variables for catalog, schema, and volume names (change, if desired)
catalog_name = "dbacademy"
schema_name = "get_started_agents"
volume_name = "customer_service"
####################################################################################

## Create Structured and Unstructured Tools

### Create a Structured Tool

Create a Unity Catalog function to retrieve customer interaction history
1. Create a function called `get_customer_interactions` that takes a customer email as input
2. The function should return interaction details including date, issue category, and description
3. Add proper comments to describe the function's purpose

In [0]:
spark.sql(f"""
---- TASK 1: Create a function to retrieve customer interaction history
---- There are multiple <FILL_IN> markings in the code below. These highlight the critical components of what are needed to create a SQL function. 
<FILL_IN> get_customer_interactions(
  customer_email <FILL_IN> 'Email address of the customer to retrieve interaction history for'
)
<FILL_IN> (
  interaction_date   DATE,
  issue_category     STRING,
  issue_description  STRING,
  customer_name      STRING
)
<FILL_IN> 'Retrieves interaction history for a specific customer including dates, categories, and descriptions'
LANGUAGE <FILL_IN>
<FILL_IN> (
  SELECT
    CAST(date_time AS DATE) as interaction_date,
    issue_category,
    issue_description,
    name as customer_name
  FROM {catalog_name}.{schema_name}.cust_service_data
  WHERE email = customer_email
  ORDER BY date_time DESC
  LIMIT 5
);
""")

If you need help, the answer is in the next cell. Click the eyeball icon to expand the cell.

In [0]:
%skip

spark.sql(f"""
---- TASK 1: Create a function to retrieve customer interaction history
CREATE OR REPLACE FUNCTION get_customer_interactions(
  customer_email STRING COMMENT 'Email address of the customer to retrieve interaction history for'
)
RETURNS TABLE (
  interaction_date   DATE,
  issue_category     STRING,
  issue_description  STRING,
  customer_name      STRING
)
COMMENT 'Retrieves interaction history for a specific customer including dates, categories, and descriptions'
LANGUAGE SQL
RETURN (
  SELECT
    CAST(date_time AS DATE) as interaction_date,
    issue_category,
    issue_description,
    name as customer_name
  FROM {catalog_name}.{schema_name}.cust_service_data
  WHERE email = customer_email
  ORDER BY date_time DESC
  LIMIT 5
);
""")

Let's test your structured tool:

In [0]:
%sql
---- Test the customer interaction function
SELECT * FROM <FILL_IN>("nicolas.pelaez@example.com");

If you need help, the answer is in the next cell. Click the eyeball icon to expand the cell.

In [0]:
%skip
%sql
-- Test the customer interaction function
SELECT * FROM get_customer_interactions("nicolas.pelaez@example.com");

### Create an Unstructured Agent Tool

Create a Unity Catalog function for product documentation search
1. Create a function called `search_product_docs` that takes a search term as input
2. The function should use vector search to find relevant product documentation
3. Return the product name and documentation excerpt

In [0]:
spark.sql(f"""
---- TASK 2: Create a vector search function for product documentation
---- There are multiple <FILL_IN> markings in the code below. These highlight the critical components of what are needed to create a SQL function. 
<FILL_IN> search_product_docs(
  search_query STRING COMMENT 'Search term for finding relevant product documentation and troubleshooting guides'
)
<FILL_IN> (
  product_name STRING,
  doc_content  STRING
)
<FILL_IN> 'Searches product documentation using vector search to find relevant troubleshooting information'
<FILL_IN>(
  SELECT
    product_name,
    indexed_doc as doc_content
  FROM
    vector_search(
      <FILL_IN> => '{catalog_name}.{schema_name}.product_docs_index',
      <FILL_IN> => search_query,
      num_results => 2
    )
);
""")

If you need help, the answer is in the next cell. Click the eyeball icon to expand the cell.

In [0]:
%skip
spark.sql(f"""
---- TASK 2: Create a vector search function for product documentation
CREATE OR REPLACE FUNCTION search_product_docs(
  search_query STRING COMMENT 'Search term for finding relevant product documentation and troubleshooting guides'
)
RETURNS TABLE (
  product_name STRING,
  doc_content  STRING
)
COMMENT 'Searches product documentation using vector search to find relevant troubleshooting information'
RETURN(
  SELECT
    product_name,
    indexed_doc as doc_content
  FROM
    vector_search(
      index => '{catalog_name}.{schema_name}.product_docs_index',
      query_vector => ai_query("databricks-gte-large-en",search_query,"STRING")
    )
);
""")

Let's test your unstructured search tool:

In [0]:
%sql
---- Test the product documentation search
SELECT 
  product_name, 
  LEFT(doc_content, 200) as doc_preview
FROM <FILL_IN>('bluetooth headphone connection');

If you need help, the answer is in the next cell. Click the eyeball icon to expand the cell.

In [0]:
%skip
%sql
---- Test the product documentation search
SELECT 
  product_name, 
  LEFT(doc_content, 200) as doc_preview
FROM search_product_docs('bluetooth headphone connection');

## Enabling Tooling in AI Playground
Now we will enable tooling in the AI Playground.
### 1. Open the AI Playground

1. In the left navigation, right-click **Playground**, and select "Open Link in New Tab".
2. Select **Meta Llamma 3.3 70b Instruct**. Any model that has the "Tools enabled" icon can be used. 

### 2. Add Tools to Your Agent
After selecting your agent, you can now add a tool in the Playground. Here is an image for reference: 

<img src="../../Includes/images/tool-select.png" alt="Tool Selection" width="600"/>


1. In the **Tools** menu select **Add**. 
1. Select **+ Add tool**.
1. Under the  **UC Function** tab, click the dropdown menu labeled **Add hosted function**.
    - Select `dbacademy.get_started_agents.search_product_docs`
    - Click on **Save**.
    - Repeat for your other function

1. **Test the Functions**
   - In the chat window, type a prompt that would require the agent to use one or more of your tools.  
     Example prompts:
     - “Show me the latest customer return.”
     - “I want to return the product I purchased recently.”
   - The agent doesn't know your account, so it will ask for your email. Provide the email for `nicolas.pelaez@example.com` as an example. When asked about the product, use `bluetooth headphone` and it should return the right order.
   - Follow the conversation and see if you are eligible for return or not.

5. **Review the Output**
   - Check the agent’s response and verify that the output matches the expected results from your functions.

6. **Compare Model Responses (Optional)**
   - You can add multiple endpoints or models to compare how different LLMs use your tools. However, please note that Databricks Free Edition places caps on the number of queries to some models.

7. **Export Your Agent (Optional)**
   - After testing, you can export your agent setup to a Python notebook for further development or deployment.
   - Navigate to **Get code** at the top of the AI Playground. 
      1. Select **Create Agent Notebook**.
      1. This will open a new window showing you pre-generated Python code. 
      1. Inspect the notebook to see what the notebook accomplishes.

## Conclusion 
You have now completed the process of building, testing, and registering agent tools in Databricks. Along the way, you created SQL functions, integrated them into the AI Playground, and validated their outputs against real-world customer service use cases. This hands-on workflow highlights how tool integration extends the power of large language models, transforming them into reliable, domain-specific assistants. Going forward, you can expand this foundation with additional functions, experiment with multiple models, and export your setup for deployment. With these skills, you're well equipped to design intelligent agents that deliver consistent and efficient results for your organization.


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> |
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> |
<a href="https://help.databricks.com/" target="blank">Support</a>